In [1]:
from Simulation.system_functions import PolymerCSTR
import numpy as np
import os
from utils.td3_helpers import load_and_prepare_system_data
from Simulation.mpc import MpcSolver, compute_observer_gain
from TD3Agent.reward_functions import make_reward_fn_mpc_quadratic, make_reward_fn_relative_QR
from utils.scaling_helpers import apply_min_max
from Simulation.mpc_run import run_mpc
from Plotting_fns.mpc_plot_fns import plot_mpc_results_cstr
import pickle

## Initialize the system

In [2]:
# First initiate the system
# Parameters
Ad = 2.142e17           # h^-1
Ed = 14897              # K
Ap = 3.816e10           # L/(molh)
Ep = 3557               # K
At = 4.50e12            # L/(molh)
Et = 843                # K
fi = 0.6                # Coefficient
m_delta_H_r = -6.99e4   # j/mol
hA = 1.05e6             # j/(Kh)
rhocp = 1506            # j/(Kh)
rhoccpc = 4043          # j/(Kh)
Mm = 104.14             # g/mol
system_params = np.array([Ad, Ed, Ap, Ep, At, Et, fi, m_delta_H_r, hA, rhocp, rhoccpc, Mm])

In [3]:
# Design Parameters
CIf = 0.5888    # mol/L
CMf = 8.6981    # mol/L
Qi = 108.       # L/h
Qs = 459.       # L/h
Tf = 330.       # K
Tcf = 295.      # K
V = 3000.       # L
Vc = 3312.4     # L

system_design_params = np.array([CIf, CMf, Qi, Qs, Tf, Tcf, V, Vc])

In [4]:
# Steady State Inputs
Qm_ss = 378.    # L/h
Qc_ss = 471.6   # L/h

system_steady_state_inputs = np.array([Qc_ss, Qm_ss])

In [5]:
# Sampling time of the system
delta_t = 0.5 # 30 mins

In [6]:
# Initiate the CSTR for steady state values
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states={"ss_inputs":cstr.ss_inputs,
               "y_ss":cstr.y_ss}

## Loading the system matrices, min max scaling, and min max of the states

In [7]:
dir_path = os.path.join(os.getcwd(), "Data")

In [8]:
# Defining the range of setpoints for data generation
setpoint_y = np.array([[2.8, 320.],
                       [5., 326.]])
u_min = np.array([71.6, 78])
u_max = np.array([870, 670])

system_data = load_and_prepare_system_data(steady_states=steady_states, setpoint_y=setpoint_y, u_min=u_min, u_max=u_max)

In [9]:
A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]

In [10]:
data_min = system_data["data_min"]
data_max = system_data["data_max"]

In [11]:
min_max_states = {'max_s': np.array([256.79686253, 256.01560603,  48.99447186, 144.79949103,
          2.82199733,   3.14014989,   2.78866348,   3.71691422,
          6.2029936 ]),
                  'min_s': np.array([ -272.28060121, -1112.33972595,   -76.63993491,  -608.60327886,
           -3.94399122,    -3.93115257,    -2.9532091 ,    -4.06547624,
          -28.25906582])}

In [12]:
y_sp_scaled_deviation = system_data["y_sp_scaled_deviation"]

In [13]:
b_min = system_data["b_min"]
b_max = system_data["b_max"]

In [14]:
min_max_dict = system_data["min_max_dict"]
min_max_dict["x_max"] = np.array([256.79686253, 256.01560603,  48.99447186, 144.79949103,
          2.82199733,   3.14014989,   2.78866348,   3.71691422,
          6.2029936 ])
min_max_dict["x_min"] = np.array([ -272.28060121, -1112.33972595,   -76.63993491,  -608.60327886,
           -3.94399122,    -3.93115257,    -2.9532091 ,    -4.06547624,
          -28.25906582])

In [15]:
# Setpoints in deviation form
inputs_number = int(B_aug.shape[1])
y_sp_scenario = np.array([[4.5, 324],
                          [3.4, 321]])

y_sp_scenario = (apply_min_max(y_sp_scenario, data_min[inputs_number:], data_max[inputs_number:])
                 - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:]))
n_tests = 2
set_points_len = 400
TEST_CYCLE = [False, False]
warm_start = 0
ACTOR_FREEZE = 0
warm_start_plot = warm_start * 2 * set_points_len + ACTOR_FREEZE

In [16]:
# Observer Gain
poles = np.array(np.array([0.44619852, 0.33547649, 0.36380595, 0.70467118, 0.3562966,
                           0.42900673, 0.4228262 , 0.96916776, 0.91230187]))
L = compute_observer_gain(A_aug, C_aug, poles)

The system is observable.


C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\Polymer_example\Simulation\mpc.py:114: UserWarning: Convergence was not reached after maxiter iterations.
You asked for a tolerance of 0.001, we got 0.9999999422182038.
  obs_gain_calc = signal.place_poles(A.T, C.T, desired_poles, method='KNV0')


# MPC Config.

In [17]:
# MPC parameters
n_inputs = 2
predict_h = 9
cont_h = 3
u_ss = apply_min_max(steady_states['ss_inputs'], data_min[:n_inputs], data_max[:n_inputs])
b_min = apply_min_max(np.array([71.6, 78]), data_min[:n_inputs], data_max[:n_inputs])
b_max= apply_min_max(np.array([870, 670]), data_min[:n_inputs], data_max[:n_inputs])
b1 = (b_min[0]-u_ss[0], b_max[0]-u_ss[0])
b2 = (b_min[1]-u_ss[1], b_max[1]-u_ss[1])
bnds = (b1, b2)*cont_h
cons = []
IC_opt = np.zeros(n_inputs*cont_h)
Q1_penalty = 5.
Q2_penalty = 1.
R1_penalty = 1
R2_penalty = 1

In [18]:
MPC_obj = MpcSolver(A_aug, B_aug, C_aug,
    Q_out=np.array([Q1_penalty, Q2_penalty]),
    R_in=np.array([R1_penalty, R2_penalty]),
    NP=predict_h,
    NC=cont_h)

## Reward configuration

In [19]:
n_inputs = 2

dy = data_max[n_inputs:] - data_min[n_inputs:]
y_sp_nom = 0.5 * (data_min[n_inputs:] + data_max[n_inputs:])

k_rel = np.array([0.003, 0.0003])
band_floor_phys = np.array([0.006, 0.07])

band_phys = np.maximum(k_rel * np.abs(y_sp_nom), band_floor_phys)

scale_factor = 1.0  # use 2.0 for [-1, 1] scaling, 1.0 for [0, 1]
band_scaled = scale_factor * band_phys / dy

q0 = 1.4
Q_diag = q0 / np.maximum(band_scaled ** 2, 1e-12)

print("dy:", dy)
print("y_sp_nom:", y_sp_nom)
print("band_phys:", band_phys)
print("band_scaled:", band_scaled)
print("Q_diag:", Q_diag)

dy: [0.22165278 0.78153727]
y_sp_nom: [  3.83915067 323.21371982]
band_phys: [0.01151745 0.09696412]
band_scaled: [0.05196169 0.12406845]
Q_diag: [518.51529284  90.95055189]


In [20]:
Q_diag = np.array([518., 90.])          # rounded from the band-based calculation
R_diag = np.array([90., 90.])          # move cost for du_scaled ~ 0.02

n_inputs = 2

print("Band scaled are:")

params, reward_fn = make_reward_fn_relative_QR(
    data_min, data_max, n_inputs,
    k_rel, band_floor_phys,
    Q_diag, R_diag,
    tau_frac=0.7,
    gamma_out=0.5, gamma_in=0.5,
    beta=7.0, gate="geom", lam_in=1.0,
    bonus_kind="exp", bonus_k=12.0, bonus_p=0.6, bonus_c=20.0,
)
print(params)

Band scaled are:
{'k_rel': array([0.003 , 0.0003]), 'band_floor_phys': array([0.006, 0.07 ]), 'band_floor_scaled': array([0.02706937, 0.08956707]), 'Q_diag': array([518.,  90.]), 'R_diag': array([90., 90.]), 'tau_frac': 0.7, 'gamma_out': 0.5, 'gamma_in': 0.5, 'beta': 7.0, 'gate': 'geom', 'lam_in': 1.0, 'bonus_kind': 'exp', 'bonus_k': 12.0, 'bonus_p': 0.6, 'bonus_c': 20.0}


In [21]:
params_qp, reward_fn_qp = make_reward_fn_mpc_quadratic(Q_diag=np.array([5., 1.]), R_diag=np.array([1., 1.]))
print(params_qp)

{'Q_diag': array([5., 1.]), 'R_diag': array([1., 1.])}


# Disturbance Profile

In [22]:
nominal_qs = 459
nominal_qi = 108
nominal_hA = 1.05e6
qi_change = 0.95
qs_change = 1.05
ha_change = 0.92

# Environment Run nominal

In [23]:
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
(y_mpc, u_mpc, avg_rewards, rewards, xhatdhat, nFE,
 time_in_sub_episodes, y_sp, yhat, delta_y_storage, delta_u_storage) = run_mpc(cstr, MPC_obj, y_sp_scenario, n_tests, set_points_len,
                                                                               steady_states, IC_opt, bnds, cons, warm_start,
                                                                               L, data_min, data_max,
                                                                               TEST_CYCLE, reward_fn_qp,
                                                                               nominal_qi, nominal_qs, nominal_hA,
                                                                               qi_change, qs_change, ha_change,
                                                                               mode="nominal")

Sub_Episode :  1  | avg. reward : -3.256941817613028
Sub_Episode :  2  | avg. reward : -4.2821273376682


# Saving the results and plotting

In [24]:
out_dir = plot_mpc_results_cstr(
    y_sp, steady_states, nFE, delta_t, time_in_sub_episodes,
    y_mpc, u_mpc, data_min, data_max,
    directory=dir_path, prefix_name="mpc_result_nominal"
)
print("Saved to:", out_dir)

Saved to: C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\Polymer_example\Data\mpc_result_nominal\20260203_153614


In [25]:
# Storing the inputs and outputs of the MPC
mpc_results = {
    "u_mpc": u_mpc,
    "y_mpc": y_mpc,
    "avg_rewards": avg_rewards,
    "xhatdhat": xhatdhat,
    "delta_y_storage": delta_y_storage,
    "delat_u_storage": delta_u_storage
}
save_path = os.path.join(dir_path, "mpc_results_nominal.pickle")
with open(save_path, 'wb') as file:
    pickle.dump(mpc_results, file)

# Environment Run Disturbance

In [23]:
cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
(y_mpc, u_mpc, avg_rewards, rewards, xhatdhat, nFE,
 time_in_sub_episodes, y_sp, yhat, delta_y_storage, delta_u_storage) = run_mpc(cstr, MPC_obj, y_sp_scenario, n_tests, set_points_len,
                                                                               steady_states, IC_opt, bnds, cons, warm_start,
                                                                               L, data_min, data_max,
                                                                               TEST_CYCLE, reward_fn_qp,
                                                                               nominal_qi, nominal_qs, nominal_hA,
                                                                               qi_change, qs_change, ha_change,
                                                                               mode="disturb")

Sub_Episode :  1  | avg. reward : -3.256941817613028
Sub_Episode :  2  | avg. reward : -4.2821273376682


# Saving the results and plotting

In [41]:
out_dir = plot_mpc_results_cstr(
    y_sp, steady_states, nFE, delta_t, time_in_sub_episodes,
    y_mpc, u_mpc, data_min, data_max,
    directory=dir_path, prefix_name="mpc_result_dist"
)
print("Saved to:", out_dir)

Saved to: C:\Users\HAMEDI\OneDrive - McMaster University\PythonProjects\Polymer_example\Data\mpc_result_dist\20260107_234051


In [37]:
# Storing the inputs and outputs of the MPC
mpc_results = {
    "u_mpc": u_mpc,
    "y_mpc": y_mpc,
    "avg_rewards": avg_rewards,
    "xhatdhat": xhatdhat,
    "delta_y_storage": delta_y_storage,
    "delat_u_storage": delta_u_storage
}
save_path = os.path.join(dir_path, "mpc_results_dist.pickle")
with open(save_path, 'wb') as file:
    pickle.dump(mpc_results, file)